In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col
import os, shutil, time
import pandas as pd
from datetime import datetime

spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("Spark siap:", spark.version)

26/11/05 07:54:27 WARN Utils: Your hostname, tarin resolves to a loopback address: 127.0.1.1; using 10.90.70.61 instead (on interface wlp0s20f3)
26/11/05 07:54:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 07:54:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/05 07:54:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark siap: 3.5.9


In [2]:
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
shutil.rmtree("tugas10_output", ignore_errors=True)
shutil.rmtree("tugas10_checkpoint", ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

print("isStreaming:", df_stream.isStreaming)

isStreaming: True


In [3]:
df_stream_total = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

agg_cabang = df_stream_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_harga"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query_cabang = agg_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Query agregasi_cabang dimulai.")

Query agregasi_cabang dimulai.


In [5]:
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Query parquet (query3) dimulai.")

Query parquet (query3) dimulai.


In [6]:
def kirim_batch(nama_file, rows):
    sekarang = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    data = {
        "pesanan_id": [r[0] for r in rows],
        "nama_cabang": [r[1] for r in rows],
        "nama_menu": [r[2] for r in rows],
        "qty": [r[3] for r in rows],
        "harga_satuan": [r[4] for r in rows],
        "waktu_pesan": [sekarang] * len(rows),
    }
    pd.DataFrame(data).to_csv(f"{folder_tugas}/{nama_file}.csv", index=False)

# Batch 1
kirim_batch("batch_1", [
    ("P1", "Magelang", "Kopi Susu", 2, 18000),
    ("P2", "Yogyakarta", "Americano", 1, 20000),
    ("P3", "Magelang", "Latte", 1, 25000),
])
print("Batch 1 dikirim")
time.sleep(8)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang").show()

Batch 1 dikirim


+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|   Magelang|      61000|             2|
| Yogyakarta|      20000|             1|
+-----------+-----------+--------------+



In [7]:
# Batch 2
kirim_batch("batch_2", [
    ("P4", "Solo", "Kopi Susu", 3, 18000),
    ("P5", "Magelang", "Americano", 2, 20000),
])
print("Batch 2 dikirim")
time.sleep(8)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang").show()

Batch 2 dikirim
+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|   Magelang|     101000|             3|
|       Solo|      54000|             1|
| Yogyakarta|      20000|             1|
+-----------+-----------+--------------+



In [8]:
# Batch 3
kirim_batch("batch_3", [
    ("P6", "Yogyakarta", "Latte", 2, 25000),
    ("P7", "Solo", "Americano", 1, 20000),
    ("P8", "Semarang", "Kopi Susu", 1, 18000),
])
print("Batch 3 dikirim")
time.sleep(8)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang").show()

Batch 3 dikirim
+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|   Magelang|     101000|             3|
|   Semarang|      18000|             1|
|       Solo|      74000|             2|
| Yogyakarta|      70000|             2|
+-----------+-----------+--------------+



In [9]:
spark.sql("""
    SELECT window.start, window.end, total_harga
    FROM agregasi_waktu
    ORDER BY window.start
""").show(truncate=False)

+-------------------+-------------------+-----------+
|start              |end                |total_harga|
+-------------------+-------------------+-----------+
|2026-11-05 07:54:00|2026-11-05 07:56:00|175000     |
|2026-11-05 07:56:00|2026-11-05 07:58:00|88000      |
+-------------------+-------------------+-----------+



In [10]:
# Kirim satu batch baru khusus untuk bukti parquet
kirim_batch("batch_4", [
    ("P9", "Magelang", "Kopi Susu", 1, 18000),
    ("P10", "Solo", "Latte", 1, 25000),
])
print("Batch 4 dikirim")
time.sleep(10)

# Bukti berkas tersimpan
!ls tugas10_output

# Baca kembali sebagai DataFrame batch biasa
df_hasil = spark.read.parquet("tugas10_output")
df_hasil.show()
print("Jumlah baris:", df_hasil.count())

Batch 4 dikirim
part-00000-3a71963a-ee8b-4fab-b8b2-2c9fe82ce068-c000.snappy.parquet
part-00000-57c04464-dad2-4efa-84f2-703ee4fd178b-c000.snappy.parquet
part-00000-5e324c3e-8854-4e52-81ea-e2c7319654c6-c000.snappy.parquet
part-00000-f36e14bb-d2bc-4e94-923a-5a9231036708-c000.snappy.parquet
_spark_metadata
+----------+-----------+---------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+-----------+---------+---+------------+-------------------+
|        P1|   Magelang|Kopi Susu|  2|       18000|2026-11-05 07:55:38|
|        P2| Yogyakarta|Americano|  1|       20000|2026-11-05 07:55:38|
|        P3|   Magelang|    Latte|  1|       25000|2026-11-05 07:55:38|
|        P6| Yogyakarta|    Latte|  2|       25000|2026-11-05 07:56:10|
|        P7|       Solo|Americano|  1|       20000|2026-11-05 07:56:10|
|        P8|   Semarang|Kopi Susu|  1|       18000|2026-11-05 07:56:10|
|        P9|   Magelang|Kopi Susu|  1|       180

In [11]:
query_cabang.stop()
query_waktu.stop()
query3.stop()
spark.stop()
print("Semua query dihentikan, SparkSession ditutup.")

Semua query dihentikan, SparkSession ditutup.
